# TPCRP / TypiClust Colab Runner

This notebook is structured to be run top-to-bottom for marking: first validate the environment, then run a quick smoke test, then build the full SimCLR cache once, then execute the baseline frameworks, and finally run the improvement experiment.


## 1. Mount Drive and clone the repo

Switch Colab to `Runtime -> Change runtime type -> GPU` before running the cells below.


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import subprocess

drive.mount("/content/drive")

REPO_URL = "https://github.com/KasimM05/Kasim-5CCSAMLF-CW2"
REPO_DIR = Path("/content/Kasim-5CCSAMLF-CW2")
CACHE_DIR = Path("/content/drive/MyDrive/tpcrp_cache")
RUNS_DIR = Path("/content/drive/MyDrive/tpcrp_runs")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repo: {REPO_DIR}")
print(f"Cache: {CACHE_DIR}")
print(f"Runs: {RUNS_DIR}")


## 2. Install dependencies


In [ ]:
%cd /content/Kasim-5CCSAMLF-CW2
%pip install -r requirements.txt


## 3. Check the Colab GPU


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not detected. Stop here and fix the runtime.")


## 4. Fast smoke test

Run this first after setup. It should finish quickly and confirm that the end-to-end pipeline works before you spend GPU time on the long runs.


In [ ]:
!python experiments/run_fully_supervised.py \
    --selector typiclust \
    --debug-subset 256 \
    --query-size 10 \
    --rounds 1 \
    --simclr-epochs 1 \
    --classifier-epochs 1 \
    --simclr-batch-size 128 \
    --classifier-batch-size 128 \
    --num-workers 2 \
    --cache-dir "$CACHE_DIR" \
    --output-dir "$RUNS_DIR/smoke/fully_supervised_typiclust"


## 5. One-time SimCLR representation cache

Run this once for the final configuration. Later framework runs reuse the same representation with `--reuse-representation --reuse-embeddings`.


In [ ]:
!python experiments/run_fully_supervised.py \
    --selector typiclust \
    --rounds 0 \
    --simclr-epochs 500 \
    --simclr-batch-size 256 \
    --classifier-batch-size 128 \
    --num-workers 2 \
    --cache-dir "$CACHE_DIR" \
    --output-dir "$RUNS_DIR/precompute/representation_only"


## 6. Fully supervised baseline matrix

This is the first full coursework run to execute after the representation cache is ready.


In [ ]:
!python tools/run_experiment_matrix.py \
    --frameworks fully_supervised \
    --cache-dir "$CACHE_DIR" \
    --output-root "$RUNS_DIR" \
    --query-size 10 \
    --rounds 5 \
    --simclr-epochs 500 \
    --classifier-epochs 20 \
    --simclr-batch-size 256 \
    --classifier-batch-size 128 \
    --num-workers 2 \
    --reuse-representation \
    --reuse-embeddings


## 7. Embedding baseline matrix

Run this after the fully supervised matrix completes successfully.


In [ ]:
!python tools/run_experiment_matrix.py \
    --frameworks embedding \
    --cache-dir "$CACHE_DIR" \
    --output-root "$RUNS_DIR" \
    --query-size 10 \
    --rounds 5 \
    --simclr-epochs 500 \
    --classifier-epochs 20 \
    --simclr-batch-size 256 \
    --classifier-batch-size 128 \
    --num-workers 2 \
    --reuse-representation \
    --reuse-embeddings


## 8. Semi-supervised baseline matrix

Run this after the embedding matrix. This is usually the slowest framework.


In [ ]:
!python tools/run_experiment_matrix.py \
    --frameworks semi_supervised \
    --cache-dir "$CACHE_DIR" \
    --output-root "$RUNS_DIR" \
    --query-size 10 \
    --rounds 5 \
    --simclr-epochs 500 \
    --semi-supervised-epochs 20 \
    --simclr-batch-size 256 \
    --classifier-batch-size 128 \
    --num-workers 2 \
    --reuse-representation \
    --reuse-embeddings


## 9. Improvement run

Only start this after the baseline matrices are complete enough for comparison in the report.


In [ ]:
!python experiments/run_improvement.py \
    --selector diversified \
    --query-size 10 \
    --rounds 5 \
    --simclr-epochs 500 \
    --classifier-epochs 20 \
    --simclr-batch-size 256 \
    --classifier-batch-size 128 \
    --num-workers 2 \
    --cache-dir "$CACHE_DIR" \
    --reuse-representation \
    --reuse-embeddings \
    --output-dir "$RUNS_DIR/improvements/diversified_full"


## 10. Inspect a run

Point this at any completed run folder to inspect the summary and per-round metrics.


In [ ]:
from pathlib import Path
import json
import pandas as pd

output_dir = RUNS_DIR / "fully_supervised" / "typiclust"
summary_path = output_dir / "run_summary.json"

if summary_path.exists():
    with summary_path.open("r", encoding="utf-8") as handle:
        summary = json.load(handle)
    print(json.dumps(summary, indent=2))
else:
    print("run_summary.json not found")

for csv_path in sorted(output_dir.glob("round_*_metrics.csv")):
    print(f"\n=== {csv_path.name} ===")
    display(pd.read_csv(csv_path))


## 11. Zip a result folder for download


In [ ]:
import shutil
from google.colab import files

target_dir = RUNS_DIR / "fully_supervised" / "typiclust"
archive_path = shutil.make_archive(str(target_dir), "zip", root_dir=str(target_dir))
files.download(archive_path)
